# Wind Blade Optimizer Demo

This notebook shows two ways to configure and run the optimizer:

1. Load a YAML config from `configs/`
2. Build a config directly in Python

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'wind_turbine_blade_moo').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from wind_turbine_blade_moo.config import (
    Config,
    DesignSpaceConfig,
    OptimizerConfig,
    ProjectConfig,
    RotorConfig,
    XfoilConfig,
    load_config,
)
from wind_turbine_blade_moo.pipeline import run_pipeline

## Option A: Load Config From YAML

Use this when you want reproducible runs from versioned config files under `configs/`.

In [ ]:
cfg_from_yaml = load_config(repo_root / 'configs/quick_test.yaml')
cfg_from_yaml = replace(
    cfg_from_yaml,
    project=replace(cfg_from_yaml.project, output_dir=(repo_root / 'outputs_notebook_demo_yaml'))
)
cfg_from_yaml

## Option B: Build Config In Python

Use this when you want to script config generation or run parameter sweeps from code.

In [ ]:
cfg_from_python = Config(
    project=ProjectConfig(output_dir=(repo_root / 'outputs_notebook_demo_python'), random_seed=123),
    rotor=RotorConfig(radius_m=35.0, wind_speed_ms=8.8, n_sections=12),
    design_space=DesignSpaceConfig(
        airfoils=['4412', '2412', '23012'],
        blades_options=[2, 3, 4],
        tip_speed_ratio_range=(5.0, 8.5),
        aoa_deg_range=(4.0, 9.0),
        hub_radius_ratio_range=(0.17, 0.26),
        chord_scale_range=(0.85, 1.20),
        twist_scale_range=(0.90, 1.12),
        chord_ratio_limits=(0.020, 0.11),
    ),
    optimizer=OptimizerConfig(population_size=18, generations=4, crossover_probability=0.9, mutation_probability=0.25),
    xfoil=XfoilConfig(
        backend='surrogate',
        fallback_to_surrogate=True,
        reynolds_bins=[250000, 900000, 2200000],
        alpha_start_deg=-4.0,
        alpha_end_deg=14.0,
        alpha_step_deg=1.0,
        max_iter=80,
        timeout_s=12.0,
    ),
)
cfg_from_python

## Run

Set `config_source` to `'yaml'` or `'python'`, then run.

In [ ]:
config_source = 'yaml'  # change to 'python' to use cfg_from_python
cfg = cfg_from_yaml if config_source == 'yaml' else cfg_from_python
summary = run_pipeline(cfg)
summary

In [ ]:
import pandas as pd

pareto = pd.read_csv(summary['outputs']['pareto_csv']).sort_values('cp', ascending=False)
best_sections = pd.read_csv(summary['outputs']['best_sections_csv'])

display(pareto.head(10))
display(best_sections[['r_over_R', 'chord_m', 'twist_deg']].head(10))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from wind_turbine_blade_moo.plotting import apply_publication_style, style_axis

apply_publication_style()

x = pareto['root_moment_nm'].to_numpy(dtype=float)
y = pareto['cp'].to_numpy(dtype=float)
sol = pareto['solidity_mean'].to_numpy(dtype=float)
obj = np.vstack([-y, x, sol]).T
ideal = obj.min(axis=0)
nadir = obj.max(axis=0)
dist = np.linalg.norm((obj - ideal) / (nadir - ideal + 1e-12), axis=1)
i_best = int(np.argmin(dist))

fig, ax = plt.subplots(figsize=(8.8, 6.2))
sc = ax.scatter(
    x,
    y,
    c=pareto['blades'],
    cmap='viridis',
    s=90 + 1600.0 * pareto['solidity_mean'],
    alpha=0.86,
    edgecolors='black',
    linewidths=0.55,
)
order = np.argsort(x)
ax.plot(x[order], np.maximum.accumulate(y[order]), color='tab:green', linewidth=3.0, label='Pareto envelope')
ax.scatter([x[i_best]], [y[i_best]], marker='*', s=300, color='tab:red', edgecolors='black', linewidths=0.8, zorder=5, label='Best compromise')
ax.annotate(
    f'Cp={y[i_best]:.3f}\nMoment={x[i_best]:,.0f} N.m',
    xy=(x[i_best], y[i_best]),
    xytext=(16, -48),
    textcoords='offset points',
    bbox=dict(boxstyle='round,pad=0.30', fc='white', ec='tab:red', alpha=0.96),
    arrowprops=dict(arrowstyle='->', color='tab:red', lw=1.3),
)

ax.set_xlabel('Root Bending Moment [N.m]')
ax.set_ylabel('Cp')
ax.set_title(f'Pareto Front ({config_source} config run)')
style_axis(ax)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Blade count')
ax.legend(loc='lower left')
plt.show()
